# VisionX — Retrieval Evaluation

This notebook evaluates the existing VisionX retrieval pipeline without changing the deployed Streamlit UI.

**Metrics:** Precision@5, Recall@5, MRR, and average search latency.

> Before calculating metrics, fill `GROUND_TRUTH` with filenames you manually judge as relevant. The CNN 84.45% result is a separate classification experiment and is not used here.


In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

SEMANTIC_METADATA_PATH = os.path.join(PROJECT_ROOT, 'Data', 'metadata_ocr_3150.csv')
SEMANTIC_EMBEDDINGS_PATH = os.path.join(PROJECT_ROOT, 'Data', 'screenshot_embeddings_3150.npy')
VISUAL_METADATA_PATH = os.path.join(PROJECT_ROOT, 'Data', 'metadata_4000.csv')
VISUAL_EMBEDDINGS_PATH = os.path.join(PROJECT_ROOT, 'Data', 'visual_embeddings_4000.npy')

print('Project root:', PROJECT_ROOT)


In [ ]:
semantic_df = pd.read_csv(SEMANTIC_METADATA_PATH)
semantic_embeddings = np.load(SEMANTIC_EMBEDDINGS_PATH)

visual_df = pd.read_csv(VISUAL_METADATA_PATH)
visual_embeddings = np.load(VISUAL_EMBEDDINGS_PATH)

print('Semantic metadata:', semantic_df.shape)
print('Semantic embeddings:', semantic_embeddings.shape)
print('Visual metadata:', visual_df.shape)
print('Visual embeddings:', visual_embeddings.shape)
print('Metadata columns:', list(semantic_df.columns))


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

semantic_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()


In [ ]:
def clean_query(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9 ]+', ' ', text)
    return re.sub(r'\s+', ' ', text)


def semantic_search(query, top_k=5):
    query_embedding = semantic_model.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, semantic_embeddings)[0]

    results = semantic_df.copy()
    results['similarity'] = similarities

    query_words = set(clean_query(query).split())

    def ocr_boost(text):
        text = clean_query(text)
        if not text:
            return 0.0
        ocr_words = set(text.split())
        if query_words.intersection(ocr_words):
            return 1.0
        if clean_query(query) in text:
            return 1.0
        return 0.0

    results['ocr_boost'] = results['ocr_text'].fillna('').apply(ocr_boost)
    results['final_score'] = results['similarity'] * 0.7 + results['ocr_boost'] * 0.3

    return results.sort_values(
        ['ocr_boost', 'final_score'],
        ascending=[False, False]
    ).head(top_k)


def clip_search(query, top_k=5):
    inputs = clip_processor(
        text=[query],
        return_tensors='pt',
        padding=True
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.inference_mode():
        output = clip_model.get_text_features(**inputs)

    query_embedding = (
        output.pooler_output
        if hasattr(output, 'pooler_output')
        else output
    )
    query_embedding = query_embedding / query_embedding.norm(
        p=2, dim=-1, keepdim=True
    )

    image_embeddings = torch.tensor(
        visual_embeddings,
        dtype=torch.float32,
        device=device
    )
    image_embeddings = image_embeddings / image_embeddings.norm(
        p=2, dim=-1, keepdim=True
    )

    similarities = (
        image_embeddings @ query_embedding.T
    ).squeeze().detach().cpu().numpy()

    results = visual_df.copy()
    results['similarity'] = similarities

    return results.sort_values(
        'similarity',
        ascending=False
    ).head(top_k)


## Step 1 — Inspect results and create ground truth

Run the next cell. For each query, inspect the top results and manually decide which filenames are relevant.

Then put those filenames into `GROUND_TRUTH`.

This manual labeling is required because Precision/Recall need known relevant results.


In [ ]:
evaluation_queries = [
    'pizza recipe',
    'hotel booking',
    'shopping product',
    'receipt',
    'address location',
    'flight ticket'
]

for query in evaluation_queries:
    print('\n' + '=' * 70)
    print('QUERY:', query)

    print('\nSEMANTIC TOP 5')
    display(
        semantic_search(query, 5)[
            ['filename', 'category', 'similarity', 'ocr_boost', 'final_score']
        ]
    )

    print('\nCLIP TOP 5')
    display(
        clip_search(query, 5)[
            ['filename', 'category', 'similarity']
        ]
    )


In [ ]:
# Replace the empty lists with filenames you manually judged as relevant.
# Use exact filenames from the result tables above.

GROUND_TRUTH = {
    'pizza recipe': [],
    'hotel booking': [],
    'shopping product': [],
    'receipt': [],
    'address location': [],
    'flight ticket': [],
}

# Example format:
# GROUND_TRUTH['pizza recipe'] = ['image1.png', 'image2.png']

print({
    query: len(filenames)
    for query, filenames in GROUND_TRUTH.items()
})


In [ ]:
def precision_at_k(results, relevant, k=5):
    if not relevant:
        return np.nan

    retrieved = set(results.head(k)['filename'])
    return len(retrieved.intersection(relevant)) / k


def recall_at_k(results, relevant, k=5):
    if not relevant:
        return np.nan

    retrieved = set(results.head(k)['filename'])
    return len(retrieved.intersection(relevant)) / len(relevant)


def reciprocal_rank(results, relevant):
    if not relevant:
        return np.nan

    relevant = set(relevant)

    for rank, filename in enumerate(results['filename'], start=1):
        if filename in relevant:
            return 1 / rank

    return 0.0


In [ ]:
def evaluate_method(search_function, method_name):
    rows = []

    for query, relevant in GROUND_TRUTH.items():
        if not relevant:
            continue

        start = time.perf_counter()
        results = search_function(query, 5)
        latency = time.perf_counter() - start

        rows.append({
            'Query': query,
            'Method': method_name,
            'Precision@5': precision_at_k(results, relevant, 5),
            'Recall@5': recall_at_k(results, relevant, 5),
            'RR': reciprocal_rank(results, relevant),
            'Latency (s)': latency
        })

    return pd.DataFrame(rows)


semantic_results = evaluate_method(
    semantic_search,
    'Semantic Search'
)

clip_results = evaluate_method(
    clip_search,
    'CLIP Visual Search'
)

evaluation_results = pd.concat(
    [semantic_results, clip_results],
    ignore_index=True
)

display(evaluation_results)


In [ ]:
summary = (
    evaluation_results
    .groupby('Method')
    .agg({
        'Precision@5': 'mean',
        'Recall@5': 'mean',
        'RR': 'mean',
        'Latency (s)': 'mean'
    })
    .rename(columns={'RR': 'MRR'})
    .reset_index()
)

display(summary)


## Interpretation

- **Precision@5:** how many of the top 5 retrieved screenshots are relevant.
- **Recall@5:** how many of the manually marked relevant screenshots were retrieved in the top 5.
- **MRR:** how early the first relevant result appears.
- **Average latency:** average search time in this evaluation environment.

Report the number of evaluation queries and the ground-truth labeling method together with these values.
